# Polars III

In this notebook, we will work with the following:

- Lazy Polars queries.
- Building a research-data pipeline as one large chain.
- Using functions to make a pipeline easier to read.
- Running and saving the final result.


In [ ]:
# imports
from pathlib import Path

import pandas as pd
import polars as pl

In [ ]:
# path
DATA_DIR = Path("..") / "data"

## Why lazy?

So far, most of our Polars work has been eager: we ask for something, and Polars does it immediately. Lazy Polars is a little different. We describe the work we want done, Polars builds a query plan, and the work runs when we call `collect()` or otherwise ask for an output.

That matters for performance. When Polars can see the whole query before running it, it can often avoid unnecessary work, read fewer columns, push filtering earlier, and choose a better way to do the operations. That is one of the reasons Polars can be very capable on ordinary research computers instead of requiring the kind of loaded workstation that felt much more necessary a few years ago.

The [Polars lazy API documentation](https://docs.pola.rs/api/python/stable/reference/lazyframe/index.html) has the full reference, but our use here will stay pretty practical.

One caveat: not every file format has a lazy scanner. Since our firm-year data are in Stata format, we'll still use pandas to read that file and convert it to Polars. After that, we'll switch to lazy work with `.lazy()`.

In [ ]:
firmyear_pd = pd.read_stata(DATA_DIR / "firmyear.dta")
firmyear_source = pl.from_pandas(firmyear_pd).lazy()

firmyear_source

## The whole pipeline as one chain

Let's rebuild the joined firm-year dataset from `1b`, but this time as a lazy pipeline. The code below works, and it has one real advantage: the full data assembly is in one place. The problem is that it is a lot to read at once.

This is a common point in real projects. A long chain is not wrong, but it can become hard to scan, explain, and revise later.

In [ ]:
lookup = pl.DataFrame(
    {
        "name": ["Google", "Microsoft"],
        "id_ticker": ["goog", "msft"],
    }
).lazy()

firmyear_pipeline_long = (
    firmyear_source.with_columns(
        pl.col("year").cast(pl.Int64),
        pl.col("count_of_employees").cast(pl.Int64),
    )
    .rename({"count_of_employees": "size_emp"})
    .sort("name", "year")
    .join(
        lookup,
        on="name",
        how="left",
        validate="m:1",
    )
    .join(
        pl.scan_csv(DATA_DIR / "stock.csv"),
        left_on=["id_ticker", "year"],
        right_on=["tic", "yr"],
        how="left",
        validate="1:1",
    )
    .join(
        pl.scan_csv(DATA_DIR / "msft_nyt.csv", try_parse_dates=True)
        .with_columns(pl.col("pub_date").dt.year().alias("year"))
        .group_by("id_ticker", "year")
        .agg(
            pl.len().alias("article_count"),
            pl.col("word_count").mean().round(1).alias("wc_mean"),
            pl.col("word_count").sum().alias("wc_sum"),
        )
        .sort("id_ticker", "year"),
        on=["id_ticker", "year"],
        how="left",
        validate="1:1",
    )
    .with_columns(
        pl.col("article_count", "wc_sum").fill_null(0),
    )
)

firmyear_pipeline_long

Notice that the previous cell did not show the actual final data. It created a `LazyFrame`, which is Polars' representation of work to be done. To run the query and get a dataframe back, we call `collect()`.

In [ ]:
firmyear_pipeline_long.collect().head()

## Making the pipeline readable

The long chain above is a good demonstration, but it is not the version I would want to maintain in a real project. We can make the same workflow easier to read by giving names to the main steps.

The functions below are not fancy. Their main job is to make the pipeline read more like the research workflow: read firm-year data, add identifiers, join stock data, summarize articles, join article summaries, and handle missing article counts.

**Aside:** The functions below use type hints, which we have not really covered yet. Type hints are not enforced by Python when the code runs. Instead, they help our tools give us better information, and they signal what kind of input a function expects and what kind of output it should return. You can read more in the [Python typing documentation](https://docs.python.org/3/library/typing.html).

In [ ]:
def firmyear_type_exprs() -> list[pl.Expr]:
    return [
        pl.col("year").cast(pl.Int64),
        pl.col("count_of_employees").cast(pl.Int64),
    ]

The function above returns expressions rather than data. That's a useful pattern when we have a small set of transformations that we may want to reuse or name clearly.

In [ ]:
def read_firmyear(path: Path) -> pl.LazyFrame:
    firmyear_pd = pd.read_stata(path)

    return (
        pl.from_pandas(firmyear_pd)
        .lazy()
        .with_columns(firmyear_type_exprs())
        .rename({"count_of_employees": "size_emp"})
        .sort("name", "year")
    )


def firm_ticker_lookup() -> pl.LazyFrame:
    return pl.DataFrame(
        {
            "name": ["Google", "Microsoft"],
            "id_ticker": ["goog", "msft"],
        }
    ).lazy()


def read_stock(path: Path) -> pl.LazyFrame:
    return pl.scan_csv(path)


def summarize_articles(path: Path) -> pl.LazyFrame:
    return (
        pl.scan_csv(path, try_parse_dates=True)
        .with_columns(pl.col("pub_date").dt.year().alias("year"))
        .group_by("id_ticker", "year")
        .agg(
            pl.len().alias("article_count"),
            pl.col("word_count").mean().round(1).alias("wc_mean"),
            pl.col("word_count").sum().alias("wc_sum"),
        )
        .sort("id_ticker", "year")
    )

Now the assembly function can focus on how the pieces fit together. This is still the same work as the long chain, but the names help us see the structure.

In [ ]:
def build_firmyear_analysis(data_dir: Path) -> pl.LazyFrame:
    return (
        read_firmyear(data_dir / "firmyear.dta")
        .join(
            firm_ticker_lookup(),
            on="name",
            how="left",
            validate="m:1",
        )
        .join(
            read_stock(data_dir / "stock.csv"),
            left_on=["id_ticker", "year"],
            right_on=["tic", "yr"],
            how="left",
            validate="1:1",
        )
        .join(
            summarize_articles(data_dir / "msft_nyt.csv"),
            on=["id_ticker", "year"],
            how="left",
            validate="1:1",
        )
        .with_columns(
            pl.col("article_count", "wc_sum").fill_null(0),
        )
    )

In [ ]:
firmyear_pipeline = build_firmyear_analysis(DATA_DIR)

firmyear_pipeline

## Running and saving

At the end of the pipeline, we collect the result and write dated output files. This is the point where the lazy query actually becomes a materialized dataframe in memory.

In [ ]:
firmyear_analysis = firmyear_pipeline.collect()

firmyear_analysis.write_csv(DATA_DIR / "2026-06-02_firmyear_pipeline_polars.csv")
firmyear_analysis.write_parquet(
    DATA_DIR / "2026-06-02_firmyear_pipeline_polars.parquet"
)

firmyear_analysis.head()

## On your own

Try making one small workflow function for a step in a project of your own, or revise one of the functions above. A good target is a function that hides low-level syntax and gives a useful name to a research-data step.

In [ ]:
# 1-1 code

In [ ]:
# 1-2 code